# Lab 0: set up your workbench

Check that the labs can talk to a model, and see what they cost. This notebook follows
[Lab 0 in the book](https://handbook.aggarwal.dev/lab-setup.html): read each step there, then run it here.
There is no lab folder for this lab: the shared adapter is `labs/common/llm.py`.

## Set up

Run the next cell first. It finds the book's code (in Colab it downloads it), installs what
the labs need and picks who answers:

- `fake` needs no key and costs nothing. Its replies are scripted, so they show how the
  code works, not what a model would say.
- `anthropic` uses Claude with your own API key. In Colab, add `ANTHROPIC_API_KEY` in the
  Secrets panel (the key icon on the left) and let this notebook read it. Elsewhere the cell
  asks for the key. It is never written into the notebook.
- `ollama` runs a small open model, free. In Colab it installs and starts Ollama for you:
  switch to a GPU first (Runtime, then Change runtime type, then T4 GPU), and expect the
  first run to download about 2 GB.

To change the backend later, change it here and run this cell again. Lab 0 in the book
explains the choices: https://handbook.aggarwal.dev/lab-setup.html

In [ ]:
# @title Set up this lab
BACKEND = "fake"  # @param ["fake", "anthropic", "ollama"]
MODEL = ""  # @param {type:"string"}

import os, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

REPO = "https://github.com/aggarwaldev/build-ai-handbook.git"


def find_root():
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "labs" / "common" / "llm.py").is_file():
            return folder
    return None


root = find_root()
if root is None:
    if not Path("build-ai-handbook").is_dir():
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", REPO], check=True)
    root = Path("build-ai-handbook").resolve()
os.chdir(root)


def run_command(*cmd):
    """Run a command from the repository root, show its output in this cell, and stop if it fails."""
    done = subprocess.run([sys.executable if part == "python" else part for part in cmd],
                          capture_output=True, text=True)
    print(done.stdout, end="")
    print(done.stderr, end="", file=sys.stderr)
    if done.returncode:
        raise RuntimeError("this command failed: " + " ".join(cmd))


if str(root / "labs") not in sys.path:
    sys.path.insert(0, str(root / "labs"))

try:
    import anthropic, numpy  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "labs/requirements.txt"], check=True)

os.environ["LLM_BACKEND"] = BACKEND
if MODEL.strip():
    os.environ["LLM_MODEL"] = MODEL.strip()
else:
    os.environ.pop("LLM_MODEL", None)
IN_COLAB = "google.colab" in sys.modules

if BACKEND == "anthropic" and not os.environ.get("ANTHROPIC_API_KEY", "").strip():
    if IN_COLAB:
        from google.colab import userdata
        try:
            os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        except Exception as err:
            raise RuntimeError("Add ANTHROPIC_API_KEY in the Secrets panel (the key icon on the left) "
                               "and allow this notebook to read it, then run this cell again.") from err
    else:
        from getpass import getpass
        os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key (not saved in the notebook): ")


def ollama_running():
    host = os.environ.get("OLLAMA_HOST", "http://localhost:11434").rstrip("/")
    try:
        urllib.request.urlopen(host + "/api/tags", timeout=2)
        return True
    except OSError:
        return False


if BACKEND == "ollama":
    if not ollama_running():
        if not IN_COLAB:
            raise RuntimeError("Start Ollama on this machine (ollama serve), or pick another backend.")
        if shutil.which("ollama") is None:
            if shutil.which("zstd") is None:
                subprocess.run("apt-get -qq update && apt-get -qq install -y zstd", shell=True, check=True)
            subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
        subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "a"), stderr=subprocess.STDOUT)
        for _ in range(60):
            if ollama_running():
                break
            time.sleep(1)
        else:
            raise RuntimeError("Ollama did not start; see ollama.log.")
    if shutil.which("nvidia-smi") is None:
        print("No GPU found: Ollama will answer slowly on the CPU.")
    pulled = os.environ.get("LLM_MODEL", "llama3.2")
    if shutil.which("ollama"):
        subprocess.run(["ollama", "pull", pulled], check=True)

print(f"Ready: code in {root}, backend {BACKEND}" + (f", model {os.environ['LLM_MODEL']}" if os.environ.get("LLM_MODEL") else ""))

## Step 1: Python and a virtual environment

The setup cell below already did this part for you: it found Python, installed the three
packages in `labs/requirements.txt` (`anthropic`, `numpy`, `pytest`) and put `labs/` on the
path, so there is no virtual environment to create or activate. This cell only shows the
Python version (the book asks for 3.10 or newer) and that the packages import.

In [ ]:
import numpy, anthropic

print("python   ", sys.version.split()[0])
print("numpy    ", numpy.__version__)
print("anthropic", anthropic.__version__)
print("repo     ", root)

## Step 2: choose who answers

The setup cell set `LLM_BACKEND` from the `BACKEND` choice at its top. Which adapter is in
use, and what answered, is what the labs print next to every reply. Look for the backend
name and the model. To switch, change `BACKEND` in the setup cell and run it again.
The [backend table](https://handbook.aggarwal.dev/lab-setup.html#backend) compares the three.

In [ ]:
from common import llm

print("LLM_BACKEND:", os.environ.get("LLM_BACKEND"))
print("adapter in use:", llm.backend_name())
print("model:", os.environ.get("LLM_MODEL") or llm.DEFAULT_MODELS[llm.backend_name()])

The labs read a few settings from the environment. Print them, hiding the key: this cell
says only whether a key is set, never what it is.
See [the settings table](https://handbook.aggarwal.dev/lab-setup.html#settings).

In [ ]:
print("LLM_BACKEND      ", os.environ.get("LLM_BACKEND", "(not set)"))
print("LLM_MODEL        ", os.environ.get("LLM_MODEL", "(not set: the backend's default)"))
print("ANTHROPIC_API_KEY", "set" if os.environ.get("ANTHROPIC_API_KEY") else "not set")
print("OLLAMA_HOST      ", os.environ.get("OLLAMA_HOST", "(not set: http://localhost:11434)"))

## Step 3: prove it works

First the tests. They run on the fake backend whatever you chose, so this costs nothing.
The last line should say how many passed and none failed.

In [ ]:
!cd labs && python -m pytest -q -p no:cacheprovider

Then one real call through the backend you chose, exactly as the page shows it. On the fake
backend the reply repeats your words. On Claude or Ollama it is a real greeting. Check that
`stop_reason` is `end` and that the `model` line names the model you meant.

In [ ]:
!python labs/01-first-call/first_call.py "Say hello in five words"

## What the labs cost

Providers charge per token, with a higher price for the tokens the model writes. The
adapter keeps the prices in `llm.PRICES_PER_MTOK`, and `llm.estimate_cost` turns token counts
into dollars. This reproduces the page's worked example, a call that sends 1,000 tokens and
gets 600 back, on each Claude model. See [what the labs cost](https://handbook.aggarwal.dev/lab-setup.html#costs).

In [ ]:
for model, (price_in, price_out) in llm.PRICES_PER_MTOK.items():
    cost = llm.estimate_cost(model, 1000, 600)
    print(f"{model:<20} ${price_in:>5.2f} in, ${price_out:>5.2f} out per million   1,000 in + 600 out = ${cost:.4f}")

The page's table gives an estimate for each lab on `claude-haiku-5-5`. Here is the idea in
code: a lab's cost is its number of calls times the cost of one call. Change the numbers
to budget your own run. Ollama and the fake backend cost nothing.

In [ ]:
calls = 30            # for example, one Lab 2 run
input_tokens = 270    # per call
output_tokens = 300   # per call, allowing for the model's thinking
per_call = llm.estimate_cost("claude-haiku-5-5", input_tokens, output_tokens)
print(f"{calls} calls on claude-haiku-5-5: ${calls * per_call:.4f}")

## Experiments

If something does not work, the book's [troubleshooting table](https://handbook.aggarwal.dev/lab-setup.html#troubleshooting)
lists what each message means. Switch `BACKEND` in the setup cell and run the cells above
again to compare the answers. Then tick the lab's checklist in the book: your progress is
kept in your browser, not in this notebook.